# Validation C — silica micro-grating cooler

This notebook compares a planar silica cooler with a one-dimensional silica grating and calculates the temperature of bare and grating-equipped silicon cells. The fast path reuses committed S4 spectra. The live rebuild is optional.

**Learning goals:** compare spectral emittance; calculate an atmospheric-window average; connect a spectral change to the cooling equilibrium; state the model-dependent mismatch.

## 1. Prepare the temporary Colab runtime

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")

def run_command(args: list[str], cwd: Path | None = None, capture: bool = False):
    print("$", shlex.join(args))
    return subprocess.run(
        args, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("Repository:", PROJECT_DIR)

## 2. Choose stored spectra or a live S4 rebuild

In [ ]:
import importlib
import importlib.util

S4_DIR = Path("/content/S4")
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"

def install_s4() -> None:
    """Build the supported phoebe-p/S4 revision in this Colab runtime."""
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable.")
        return
    run_command(["apt-get", "-qq", "update"])
    run_command([
        "apt-get", "-qq", "install", "-y", "build-essential", "git",
        "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
        "libopenblas-dev", "libsuitesparse-dev",
    ])
    if not S4_DIR.exists():
        run_command(["git", "clone", "https://github.com/phoebe-p/S4.git", str(S4_DIR)])
    run_command(["git", "checkout", S4_COMMIT], cwd=S4_DIR)
    run_command(["make", "-j2", "S4_pyext"], cwd=S4_DIR)
    importlib.invalidate_caches()
    import S4
    print("S4:", S4.__file__)

In [ ]:
REBUILD_S4 = False
if REBUILD_S4:
    install_s4()

command = [sys.executable, "run_validation.py"]
if not REBUILD_S4:
    command.append("--no-build")
run_command(command, cwd=PROJECT_DIR / "validations/validation C")

## 3. Plot and quantify the 8–13 µm emittance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data_dir = PROJECT_DIR / "validations/validation C/data/optics"
planar = np.loadtxt(data_dir / "cooler_planar.txt")
grating = np.loadtxt(data_dir / "cooler_grating.txt")

mask_planar = (planar[:, 0] >= 8.0) & (planar[:, 0] <= 13.0)
mask_grating = (grating[:, 0] >= 8.0) & (grating[:, 0] <= 13.0)
planar_mean = np.trapezoid(planar[mask_planar, 3], planar[mask_planar, 0]) / 5.0
grating_mean = np.trapezoid(grating[mask_grating, 3], grating[mask_grating, 0]) / 5.0
print(f"Planar window emittance:  {planar_mean:.3f}")
print(f"Grating window emittance: {grating_mean:.3f}")

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(planar[:, 0], planar[:, 3], label="Planar silica")
ax.plot(grating[:, 0], grating[:, 3], label="Silica grating")
ax.set(xlim=(8.0, 13.0), ylim=(0.0, 1.02),
       xlabel="Wavelength (µm)", ylabel="Normal emittance")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.show()

## 4. Interpret and extend

The calculated grating window emittance is 0.938, compared with approximately 0.90 in the paper. Its temperature rise is 37.8 °C versus 37.5 °C in the paper. The bare-cell result is poor: 93.9 °C versus 77.5 °C. This is therefore a **partial validation**, not a general validation of the temperature model. The fixed solar absorptance, angle-independent use of normal optics, and silicon optical model matter.

**Exercise:** edit the grating duty cycle in a copied optics YAML. Before doing a live rebuild, reduce the wavelength count for a smoke test; then converge a named observable before making a scientific claim.

**Reference:** B. Zhao et al., *Renewable Energy* 191, 662–668 (2022), [doi:10.1016/j.renene.2022.04.063](https://doi.org/10.1016/j.renene.2022.04.063).